# 作业4-5：CNN、RNN、Transformer 及 AI 辅助编程练习

## 1. 卷积神经网络

### 1.1 目标

通过对 MNIST 数据进行训练，构建一个简单的图像分类模型，对图片中的数字进行识别。你将利用该模型对自己真实手写出的数字进行预测，观察模型效果。

### 1.2 主要步骤

1. 获取数据
2. 定义模型结构
3. 创建模型类
4. 定义损失函数
5. 编写训练循环
6. 实施预测

### 1.3 获取数据

我们使用知名的 MNIST 数据集，它可以从 PyTorch 中利用工具函数下载得到。MNIST 数据训练集大小为60000，我们将**使用完整训练集进行训练**，并对10个测试集观测进行预测展示。以下函数会在当前目录建立一个名为 data 的文件夹，其中会包含下载得到的数据集。

**注意：请在任何程序的最开始加上随机数种子的设置。请保持这一习惯。**

In [ ]:
import numpy as np
import torch
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader

np.random.seed(123456)
torch.manual_seed(123456)

mnist = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)
loader = DataLoader(mnist, batch_size=60000, shuffle=True)

我们一次性取出60000个观测，其中 x 是图片数据，y 是图片对应的数字。

In [ ]:
x, y = next(iter(loader))

一个习惯性动作是查看数据的大小和维度。

In [ ]:
print(x.shape)
print(y.shape)

用类似的方法获取测试集，并取出10个观测：

In [ ]:
mnist_test = datasets.MNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)
loader = DataLoader(mnist_test, batch_size=10, shuffle=True)

xtest, ytest = next(iter(loader))
print(xtest.shape)
print(ytest.shape)

我们可以利用下面的函数展示图片的内容。如选择第一张测试图片，先将其转换成 Numpy 数组，再绘制图形：

In [ ]:
import matplotlib.pyplot as plt

img = xtest[0].squeeze().cpu().numpy()
print(img.shape)
plt.imshow(img, cmap="gray")
plt.show()

接下来请你选择5个你喜欢的数字（60000以下），然后取出训练集中对应位置的图片，并画出它们的内容。

In [ ]:
# 在此处完成代码
# 选择5个喜欢的数字（60000以下）
indices = [100, 5000, 15000, 30000, 55000]

# 取出训练集中对应位置的图片并画出它们的内容
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, idx in enumerate(indices):
    img = x[idx].squeeze().cpu().numpy()
    axes[i].imshow(img, cmap="gray")
    axes[i].set_title(f"Label: {y[idx].item()}")
    axes[i].axis('off')
plt.show()

### 1.4 定义模型结构

我们搭建一个类似于 LeNet-5 的网络，结构如下：

![](https://pic1.zhimg.com/80/v2-82eabb4c17e90d467197d013f7629f3c_720w.jpg)

我们需要创建2个卷积层、2个汇聚层和2个全连接层，**暂时忽略所有的激活函数**。所有隐藏层的函数细节都可以在[官方文档](https://pytorch.org/docs/stable/nn.html)中按分类找到。每一个隐藏层本质上都是将一个数组变换成另一个数组的函数，因此为了确认编写的模型是正确的，可以先用一个小数据进行测试，观察输入和输出的维度。例如，我们先取出前6个观测，此时输入的维度是 `[6, 1, 28, 28]`：

In [ ]:
ns = 6
smallx = x[0:ns]
smally = y[0:ns]
print(smallx.shape)
print(smally.shape)

接下来创建第1个卷积层，并测试输出的维度。注意到我们可以直接将隐藏层当成一个函数来调用。

In [ ]:
conv1 = torch.nn.Conv2d(in_channels=1, out_channels=20, kernel_size=5, stride=1)
res1 = conv1(smallx)
print(res1.shape)

可以看到，输出的维度为 `[20, 24, 24]`（不包括第1位的数据批次维度），与之前图中的结果吻合。

接下来，请按照图中提示编写层对象 `pool1`、`conv2`、`pool2`、`fc1` 和 `fc2`，并顺次测试输入与输出的维度，使其与上图匹配。注意，在将一个大小为 `[6, 50, 4, 4]` 的数组（假设叫 `somearray`）传递给 `fc1` 之前，需要先将其变形为只有两个维度的数组，做法是 `somearray.view(-1, 50 * 4 * 4)`，其中 -1 表示该位置的大小不变。也可以使用 `torch.flatten()` 函数并指定其中的 `start_dim` 参数（请搜索其对应的函数文档）。

```py
pool1 = ...
res2 = pool1(res1)
print(res2.shape)
assert res2.shape == (ns, 20, 12, 12), "pool1 输出形状不对"

conv2 = ...
res3 = conv2(res2)
print(res3.shape)
assert res3.shape == (ns, 50, 8, 8), "conv2 输出形状不对"

pool2 = ...
res4 = pool2(res3)
print(res4.shape)
assert res4.shape == (ns, 50, 4, 4), "pool2 输出形状不对"

fc1 = ...
res5 = fc1(res4.view(-1, 800))
print(res5.shape)
assert res5.shape == (ns, 500), "fc1 输出形状不对"

fc2 = ...
res6 = fc2(res5)
print(res6.shape)
assert res6.shape == (ns, 10), "fc2 输出形状不对"
```

In [ ]:
# 在此处完成代码
import torch.nn.functional as F

# 创建各个层对象
pool1 = torch.nn.MaxPool2d(kernel_size=2, stride=2)
res2 = pool1(res1)
print(res2.shape)
assert res2.shape == (ns, 20, 12, 12), "pool1 输出形状不对"

conv2 = torch.nn.Conv2d(in_channels=20, out_channels=50, kernel_size=5, stride=1)
res3 = conv2(res2)
print(res3.shape)
assert res3.shape == (ns, 50, 8, 8), "conv2 输出形状不对"

pool2 = torch.nn.MaxPool2d(kernel_size=2, stride=2)
res4 = pool2(res3)
print(res4.shape)
assert res4.shape == (ns, 50, 4, 4), "pool2 输出形状不对"

fc1 = torch.nn.Linear(in_features=50 * 4 * 4, out_features=500)
res5 = fc1(res4.view(-1, 50 * 4 * 4))
print(res5.shape)
assert res5.shape == (ns, 500), "fc1 输出形状不对"

fc2 = torch.nn.Linear(in_features=500, out_features=10)
res6 = fc2(res5)
print(res6.shape)
assert res6.shape == (ns, 10), "fc2 输出形状不对"

### 1.5 创建模型类

在确保隐藏层维度都正确后，将所有的隐藏层封装到一个模型类中，其中模型结构在 `__init__()` 中定义，具体的计算过程在 `forward()` 中实现。此时需要加入激活函数。在本模型中，**请在 `conv1`、`conv2` 和 `fc1` 后加入 ReLU 激活函数，并在 `fc2` 后加入 Softmax 激活函数**。

```py
class MyModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = ...
        self.pool1 = ...
        self.conv2 = ...
        self.pool2 = ...
        self.fc1 = ...
        self.fc2 = ...

    def forward(self, x):
        x = self.conv1(x)
        x = torch.relu(x)
        x = self.pool1(x)
        ...
        return x
```

In [ ]:
# 在此处完成代码
class MyModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = torch.nn.Conv2d(in_channels=1, out_channels=20, kernel_size=5, stride=1)
        self.pool1 = torch.nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = torch.nn.Conv2d(in_channels=20, out_channels=50, kernel_size=5, stride=1)
        self.pool2 = torch.nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = torch.nn.Linear(in_features=50 * 4 * 4, out_features=500)
        self.fc2 = torch.nn.Linear(in_features=500, out_features=10)

    def forward(self, x):
        x = self.conv1(x)
        x = torch.relu(x)
        x = self.pool1(x)
        x = self.conv2(x)
        x = torch.relu(x)
        x = self.pool2(x)
        x = x.view(-1, 50 * 4 * 4)
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.fc2(x)
        x = torch.softmax(x, dim=1)
        return x

再次测试输入输出的维度是否正确。如果模型编写正确，输出的维度应该是 `[6, 10]`，且输出结果为0到1之间的概率值。

In [ ]:
np.random.seed(123)
torch.manual_seed(123)

model = MyModel()
pred = model(smallx)
print(pred.shape)
print()
print(pred)
print()
print(torch.sum(pred, dim=1))

`pred` 的每一行加总为1，其中每一个元素代表对应类别的预测概率。

我们还可以直接打印模型对象，观察隐藏层的结构：

In [ ]:
print(model)

### 1.6 定义损失函数

对于分类问题，损失函数通常选取为负对数似然函数。在 PyTorch 中，可以使用 `torch.nn.NLLLoss` 来完成计算。其用法是先定义一个损失函数对象，然后在预测值和真实标签上调用该函数对象。注意：损失函数对象的第一个参数是预测概率的**对数值**，第二个参数是真实的标签。[文档说明](https://pytorch.org/docs/stable/generated/torch.nn.NLLLoss.html)。

In [ ]:
lossfn = torch.nn.NLLLoss()
lossfn(torch.log(pred), smally)

### 1.7 编写训练循环

对于本数据，选取 mini-batch 大小为200，共遍历数据3遍，优化器选为 SGD，学习率为0.001。记录每个 mini-batch 下的损失函数值存放到列表 `losses_sgd` 中，然后画出损失函数的曲线。

In [ ]:
nepoch = 3
batch_size = 200
lr = 0.001

np.random.seed(123)
torch.manual_seed(123)

model = MyModel()
losses_sgd = []

# 创建数据加载器
train_loader = DataLoader(mnist, batch_size=batch_size, shuffle=True)

# 定义优化器和损失函数
optimizer = torch.optim.SGD(model.parameters(), lr=lr)
loss_fn = torch.nn.NLLLoss()

# 训练循环
for epoch in range(nepoch):
    for batch_idx, (data, target) in enumerate(train_loader):
        # 前向传播
        output = model(data)
        loss = loss_fn(torch.log(output), target)
        
        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # 记录损失
        losses_sgd.append(loss.item())
        
print(f"SGD训练完成，共{len(losses_sgd)}个损失值")

In [ ]:
plt.plot(losses_sgd)

接下来使用 Adagrad 优化器（在[官方文档](https://pytorch.org/docs/stable/optim.html)中找到对应的函数），其他参数保持不变，重新训练一次模型，也保存下来损失函数值。

In [ ]:
nepoch = 3
batch_size = 200
lr = 0.001

np.random.seed(123)
torch.manual_seed(123)

model = MyModel()
losses_adagrad = []

# 创建数据加载器
train_loader = DataLoader(mnist, batch_size=batch_size, shuffle=True)

# 定义优化器和损失函数
optimizer = torch.optim.Adagrad(model.parameters(), lr=lr)
loss_fn = torch.nn.NLLLoss()

# 训练循环
for epoch in range(nepoch):
    for batch_idx, (data, target) in enumerate(train_loader):
        # 前向传播
        output = model(data)
        loss = loss_fn(torch.log(output), target)
        
        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # 记录损失
        losses_adagrad.append(loss.item())
        
print(f"Adagrad训练完成，共{len(losses_adagrad)}个损失值")

对比 SGD 和 Adagrad，画出各自的损失函数曲线。

In [ ]:
plt.plot(losses_sgd)
plt.plot(losses_adagrad)

最后再自行选择一款优化器，重复上面的实验，并画出三种优化器的损失函数值对比图。

In [ ]:
nepoch = 3
batch_size = 200
lr = 0.001

np.random.seed(123)
torch.manual_seed(123)

model = MyModel()
losses_adam = []

# 创建数据加载器
train_loader = DataLoader(mnist, batch_size=batch_size, shuffle=True)

# 定义优化器和损失函数
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
loss_fn = torch.nn.NLLLoss()

# 训练循环
for epoch in range(nepoch):
    for batch_idx, (data, target) in enumerate(train_loader):
        # 前向传播
        output = model(data)
        loss = loss_fn(torch.log(output), target)
        
        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # 记录损失
        losses_adam.append(loss.item())
        
print(f"Adam训练完成，共{len(losses_adam)}个损失值")

# 对比三种优化器的损失函数值
plt.figure(figsize=(12, 6))
plt.plot(losses_sgd, label='SGD')
plt.plot(losses_adagrad, label='Adagrad')
plt.plot(losses_adam, label='Adam')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.legend()
plt.title('Comparison of Optimizers')
plt.show()

### 1.8 实施预测

为了验证模型的效果，我们对10个测试观测（即之前生成的 `testx`）进行预测。

In [ ]:
ypred = model(xtest)
print(np.round(ypred.detach().cpu().numpy(), 3))
print(ytest)

如果模型搭建和训练都正常，那么每一行中概率最大的取值所在的位置应该正好对应真实的标签。我们也可以让 PyTorch 自动找到最大值的位置。

In [ ]:
torch.argmax(ypred, dim=1)

最后，我们用模型对一些真实的手写数字图片进行预测。请你利用绘图软件（如 Windows 自带的绘图，或 Photoshop 等）准备10张正方形黑色底色的图片，每张用鼠标绘制一个数字（**请使用较粗的笔划**），从0到9，然后以0.png，1.png等文件名存储下来，放到当前目录一个名为 digits 的文件夹中。以下是几个例子：
![](digits/sample0.png) ![](digits/sample5.png) ![](digits/sample8.png)

接下来利用 Pillow 软件包读取图片：

In [ ]:
from PIL import Image
im = Image.open(r"C:\Users\19716\Downloads\HW4-5 (2)\digits\sample0.png")
im

此时如果直接将其转为 Numpy 数组会得到三个或四个通道（可能有一个透明度通道）：

In [ ]:
im_arr = np.array(im)
print(im_arr.shape)

因此，我们先强制转换为灰度图片（单通道），再缩放至模型的图片大小 28 x 28：

In [ ]:
im = im.convert("L")
im.thumbnail((28, 28))
im_arr = np.array(im)
print(im_arr.shape)
im

为了传递给模型对象，还需要先将数值归一化到 [0,1] 区间，转换为 PyTorch 的 Tensor 类型，并增加一个批次和一个通道的维度：

In [ ]:
test0 = torch.tensor(im_arr / 255.0, dtype=torch.float32).view(1, 1, 28, 28)
print(test0.shape)

最后对图片标签进行预测：

In [ ]:
pred0 = model(test0)
print(np.round(pred0.detach().cpu().numpy(), 3))

预测结果是否符合真实情形？请对你自己绘制出的10张图片进行类似的预测操作，并评价其效果。

In [ ]:
# 在此处完成代码
# 对自己绘制的10张图片进行预测操作
from PIL import Image
import os

# 检查digits文件夹是否存在
digits_folder = r'C:\Users\19716\Desktop\digits'
if os.path.exists(digits_folder):
    print("找到digits文件夹，开始处理自定义手写数字图片...")
    
    # 创建图表显示预测结果
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    axes = axes.ravel()
    
    predicted_labels = []
    
    # 遍历0-9的图片文件
    for i in range(10):
        filename = f"{i}.png"
        filepath = os.path.join(digits_folder, filename)
        
        if os.path.exists(filepath):
            try:
                # 读取图片
                im = Image.open(filepath)
                
                # 预处理图片
                im = im.convert("L")  # 转换为灰度图
                im.thumbnail((28, 28))  # 缩放至28x28
                im_arr = np.array(im)
                
                # 归一化并转换为PyTorch张量
                test_img = torch.tensor(im_arr / 255.0, dtype=torch.float32).view(1, 1, 28, 28)
                
                # 进行预测
                pred = model(test_img)
                pred_label = torch.argmax(pred, dim=1).item()
                confidence = torch.max(pred).item()
                
                predicted_labels.append((i, pred_label, confidence))
                
                # 显示图片和预测结果
                axes[i].imshow(im_arr, cmap='gray')
                axes[i].set_title(f"Original: {i}\nPredicted: {pred_label}\nConfidence: {confidence:.2f}")
                axes[i].axis('off')
            except Exception as e:
                print(f"处理文件 {filename} 时出错: {e}")
                axes[i].set_title(f"Error: {i}")
                axes[i].axis('off')
        else:
            print(f"未找到文件: {filepath}")
            axes[i].set_title(f"Missing: {i}")
            axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # 打印预测结果总结
    correct = 0
    print("\n预测结果总结:")
    print("原始标签 -> 预测标签 (置信度)")
    print("-" * 30)
    for original, predicted, confidence in predicted_labels:
        status = "✓" if original == predicted else "✗"
        print(f"{original} -> {predicted} ({confidence:.2f}) {status}")
        if original == predicted:
            correct += 1
    
    accuracy = correct / len(predicted_labels) if predicted_labels else 0
    print(f"\n准确率: {accuracy:.2%} ({correct}/{len(predicted_labels)})")
    
    # 评价效果
    print("\n模型效果评价:")
    if accuracy >= 0.8:
        print("模型表现优秀，能够较好地识别手写数字。")
    elif accuracy >= 0.6:
        print("模型表现良好，但对于某些手写样式可能存在识别困难。")
    else:
        print("模型表现一般，可能需要更多的训练或改进预处理方法。")
else:
    print("未找到digits文件夹，请确保已按照要求创建并放置手写数字图片。")

## 2. 循环神经网络

以 `names.txt` 中的英文名作为训练集，利用 RNN 或 LSTM 等方法对字母序列数据进行建模，每个字母视为序列中的一个元素，然后使用拟合的模型随机生成50个名字。本练习为开放式，不指定各类超参数（如网络结构、学习率、迭代次数等），但需提供必要的输出和诊断结果支持你的选择（如模型是否收敛、效果评价等）。

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import random
import string

# 设置随机种子
np.random.seed(123456)
torch.manual_seed(123456)

# 读取names.txt文件
with open(r'C:\Users\19716\Desktop\names.txt', 'r', encoding='utf-8') as f:
    names = [line.strip() for line in f if line.strip()]

print(f"总共读取到 {len(names)} 个名字")
print("前10个名字:", names[:10])

# 构建字符词汇表
all_letters = string.ascii_letters + " .,;'"
n_letters = len(all_letters) + 1  # 加1表示结束符

# 字符到索引的映射
char_to_idx = {char: idx for idx, char in enumerate(all_letters)}
char_to_idx['<EOS>'] = n_letters - 1  # 结束符

# 索引到字符的映射
idx_to_char = {idx: char for char, idx in char_to_idx.items()}

# 创建更智能的名称生成函数
valid_chars = [c for c in all_letters if c not in [' ', '.', ',', ';', "'"]]

def generate_name(model, max_length=20):
    """生成一个新的名字，确保至少有一个字符"""
    with torch.no_grad():
        # 选择一个有效的起始字符
        start_char = random.choice(valid_chars)
        input_tensor = torch.zeros(1, 1, n_letters)
        input_tensor[0][0][char_to_idx[start_char]] = 1
        hidden = model.init_hidden()
        name = start_char
        
        # 生成剩余字符
        for i in range(max_length-1):
            output, hidden = model(input_tensor, hidden)
            letter, letter_idx = sample_letter(output[0][0])
            
            if letter == '<EOS>' or i == max_length-2:
                break
            name += letter
            
            # 准备下一个输入
            input_tensor = torch.zeros(1, 1, n_letters)
            input_tensor[0][0][letter_idx] = 1
        
        # 如果生成了空字符串，返回一个默认值
        return name if name else random.choice(names)

# 修改采样函数，避免陷入循环
def sample_letter(output):
    """从输出分布中采样一个字母"""
    probabilities = torch.softmax(output, dim=0)
    
    # 确保不会总是采样到<EOS>
    if char_to_idx['<EOS>'] < len(probabilities) and probabilities[char_to_idx['<EOS>']] > 0.8:
        # 降低结束符的概率
        probabilities[char_to_idx['<EOS>']] *= 0.5
        # 重新归一化
        probabilities = probabilities / probabilities.sum()
    
    sampled_idx = torch.multinomial(probabilities, 1).item()
    return idx_to_char[sampled_idx], sampled_idx

# 定义RNN模型
class NameRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, n_layers=1):
        super(NameRNN, self).__init__()
        self.hidden_size = hidden_size
        self.n_layers = n_layers
        
        self.rnn = nn.RNN(input_size, hidden_size, n_layers, batch_first=False)
        self.fc = nn.Linear(hidden_size, output_size)
        
    def forward(self, input, hidden):
        output, hidden = self.rnn(input, hidden)
        output = self.fc(output)
        return output, hidden
    
    def init_hidden(self, batch_size=1):
        return torch.zeros(self.n_layers, batch_size, self.hidden_size)

# 设置超参数
hidden_size = 128
n_layers = 1
learning_rate = 0.005
epochs = 50

# 初始化模型
rnn_model = NameRNN(n_letters, hidden_size, n_letters, n_layers)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(rnn_model.parameters(), lr=learning_rate)

# 训练模型
losses_rnn = []
for epoch in range(epochs):
    total_loss = 0
    for name in names[:1000]:  # 使用前1000个名字进行训练以节省时间
        if len(name) < 2:  # 跳过太短的名字
            continue
        
        # 准备输入和目标
        input_tensor = name_to_tensor(name)
        target_tensor = target_to_tensor(name)
        
        # 初始化隐藏状态
        hidden = rnn_model.init_hidden()
        
        # 清零梯度
        optimizer.zero_grad()
        
        # 前向传播
        output, hidden = rnn_model(input_tensor, hidden)
        
        # 计算损失
        loss = criterion(output.squeeze(1), target_tensor.squeeze(1))
        
        # 反向传播
        loss.backward()
        
        # 更新参数
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(names[:1000])
    losses_rnn.append(avg_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}')

# 绘制损失曲线
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
plt.plot(losses_rnn)
plt.title('RNN Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

# 生成50个新名字
print("\n生成的50个新名字:")
generated_names_rnn = []
for i in range(50):
    name = generate_name(rnn_model)
    generated_names_rnn.append(name)
    print(f"{i+1:2d}: {name}")

## 3. Transformer

利用 Transformer 类型的网络架构，同样对 `names.txt` 中的英文名进行序列建模和训练，再使用拟合的模型随机生成50个名字。

In [ ]:
# 在此处完成代码
import torch
import torch.nn as nn
import math

# 定义位置编码
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return x

# 定义Transformer模型
class NameTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=8, num_layers=2, max_len=50):
        super(NameTransformer, self).__init__()
        self.d_model = d_model
        self.max_len = max_len
        
        # 嵌入层
        self.embedding = nn.Embedding(vocab_size, d_model)
        # 位置编码
        self.pos_encoder = PositionalEncoding(d_model, max_len)
        # Transformer解码器层
        decoder_layer = nn.TransformerDecoderLayer(d_model, nhead, batch_first=False)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers)
        # 输出层
        self.fc_out = nn.Linear(d_model, vocab_size)
        # 掩码
        self.mask = self.generate_square_subsequent_mask(max_len)
        
    def generate_square_subsequent_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask
        
    def forward(self, src, tgt=None):
        # 如果没有提供tgt，则使用src作为tgt（用于生成）
        if tgt is None:
            tgt = src[:-1]  # 去掉最后一个字符
            
        # 嵌入和位置编码
        tgt_emb = self.embedding(tgt) * math.sqrt(self.d_model)
        tgt_emb = self.pos_encoder(tgt_emb)
        
        # Transformer解码器
        if tgt.size(0) <= self.max_len:
            output = self.transformer_decoder(tgt_emb, src, tgt_mask=self.mask[:tgt.size(0), :tgt.size(0)])
        else:
            # 如果序列太长，使用无掩码
            output = self.transformer_decoder(tgt_emb, src)
            
        # 输出层
        output = self.fc_out(output)
        return output

# 修改数据处理函数以适应Transformer
def name_to_indices(name):
    """将名字转换为索引序列"""
    indices = [char_to_idx.get(char, char_to_idx['<EOS>']) for char in name]
    indices.append(char_to_idx['<EOS>'])  # 添加结束符
    return indices

def indices_to_tensor(indices):
    """将索引序列转换为张量"""
    return torch.LongTensor(indices).unsqueeze(1)  # 添加批次维度

# 准备训练数据
train_names = names[:1000]  # 使用前1000个名字进行训练以节省时间
train_data = [name_to_indices(name) for name in train_names]

# 设置超参数
vocab_size = n_letters
d_model = 128
nhead = 8
num_layers = 2
learning_rate = 0.001
epochs = 50

# 初始化模型
transformer_model = NameTransformer(vocab_size, d_model, nhead, num_layers)
criterion_transformer = nn.CrossEntropyLoss()
optimizer_transformer = torch.optim.Adam(transformer_model.parameters(), lr=learning_rate)

# 创建用于Transformer的虚拟编码器输出（这里简化处理）
encoder_output = torch.randn(20, 1, d_model)  # 假设最大长度为20

# 训练Transformer模型
losses_transformer = []
for epoch in range(epochs):
    total_loss = 0
    for name_indices in train_data:
        if len(name_indices) < 2:  # 至少需要2个字符才能形成输入-目标对
            continue
            
        # 准备输入和目标
        input_indices = name_indices[:-1]  # 输入是除了最后一个字符的所有字符
        target_indices = name_indices[1:]  # 目标是除了第一个字符的所有字符
        
        input_tensor = indices_to_tensor(input_indices)
        target_tensor = torch.LongTensor(target_indices)
        
        # 清零梯度
        optimizer_transformer.zero_grad()
        
        # 前向传播
        output = transformer_model(encoder_output[:len(input_indices)], input_tensor)
        
        # 计算损失
        loss = criterion_transformer(output.squeeze(1), target_tensor)
        
        # 反向传播
        loss.backward()
        
        # 更新参数
        optimizer_transformer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_data)
    losses_transformer.append(avg_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f'Transformer Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}')

# 绘制Transformer损失曲线
plt.figure(figsize=(10, 5))
plt.plot(losses_transformer)
plt.title('Transformer Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

# 生成新名字的函数（Transformer版本）
def generate_name_transformer(model, max_length=20):
    """使用Transformer生成一个新的名字"""
    with torch.no_grad():
        # 从开始符开始
        name_indices = [char_to_idx['<EOS>']]
        
        for i in range(max_length):
            # 准备输入
            input_tensor = indices_to_tensor(name_indices)
            
            # 前向传播
            output = model(encoder_output[:len(name_indices)], input_tensor)
            
            # 从最后一个输出中采样
            last_output = output[-1, 0, :]
            probabilities = torch.softmax(last_output, dim=0)
            sampled_idx = torch.multinomial(probabilities, 1).item()
            
            # 如果采样到结束符，则停止
            if sampled_idx == char_to_idx['<EOS>']:
                break
                
            name_indices.append(sampled_idx)
        
        # 将索引转换回字符
        name = ''.join([idx_to_char[idx] for idx in name_indices[1:]])  # 跳过开始符
        return name

# 生成50个新名字
print("\n使用Transformer生成的50个新名字:")
generated_names_transformer = []
for i in range(50):
    name = generate_name_transformer(transformer_model)
    generated_names_transformer.append(name)
    print(f"{i+1:2d}: {name}")

## 4. AI 辅助编程工具实战

# 在此处完成代码
# 创建一个简单的猜数字游戏HTML页面
html_code = '''
<!DOCTYPE html>
<html lang="zh-CN">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>猜数字游戏</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            text-align: center;
            margin-top: 50px;
            background-color: #f0f0f0;
        }
        .container {
            max-width: 500px;
            margin: 0 auto;
            padding: 20px;
            background-color: white;
            border-radius: 10px;
            box-shadow: 0 0 10px rgba(0,0,0,0.1);
        }
        input, button {
            padding: 10px;
            margin: 10px;
            font-size: 16px;
        }
        button {
            background-color: #4CAF50;
            color: white;
            border: none;
            cursor: pointer;
            border-radius: 5px;
        }
        button:hover {
            background-color: #45a049;
        }
        #result {
            font-size: 18px;
            font-weight: bold;
            margin: 20px 0;
        }
        .attempt {
            color: #ff9800;
        }
        .success {
            color: #4CAF50;
        }
        .error {
            color: #f44336;
        }
    </style>
</head>
<body>
    <div class="container">
        <h1>猜数字游戏</h1>
        <p>我想了一个1到100之间的数字，你能猜出来吗？</p>
        <input type="number" id="guessInput" min="1" max="100" placeholder="输入你的猜测">
        <br>
        <button onclick="checkGuess()">猜!</button>
        <button onclick="resetGame()">重新开始</button>
        <div id="result"></div>
        <div id="attempts"></div>
    </div>

    <script>
        // 初始化游戏变量
        let secretNumber = Math.floor(Math.random() * 100) + 1;
        let attempts = 0;
        let maxAttempts = 10;

        function checkGuess() {
            // 获取用户输入
            const guessInput = document.getElementById('guessInput');
            const resultDiv = document.getElementById('result');
            const attemptsDiv = document.getElementById('attempts');
            
            const userGuess = parseInt(guessInput.value);
            
            // 验证输入
            if (isNaN(userGuess) || userGuess < 1 || userGuess > 100) {
                resultDiv.innerHTML = '<span class="error">请输入1到100之间的有效数字!</span>';
                return;
            }
            
            // 增加尝试次数
            attempts++;
            
            // 检查猜测
            if (userGuess === secretNumber) {
                resultDiv.innerHTML = `<span class="success">恭喜你! 数字就是 ${secretNumber}!</span>`;
                attemptsDiv.innerHTML = `你用了 ${attempts} 次猜中了数字!`;
                guessInput.disabled = true;
            } else if (attempts >= maxAttempts) {
                resultDiv.innerHTML = `<span class="error">游戏结束! 正确答案是 ${secretNumber}.</span>`;
                guessInput.disabled = true;
            } else if (userGuess < secretNumber) {
                resultDiv.innerHTML = '<span class="attempt">太小了! 再试一次。</span>';
            } else {
                resultDiv.innerHTML = '<span class="attempt">太大了! 再试一次。</span>';
            }
            
            // 显示剩余尝试次数
            if (attempts < maxAttempts && userGuess !== secretNumber) {
                attemptsDiv.innerHTML = `已尝试: ${attempts}/${maxAttempts} 次`;
            }
            
            // 清空输入框
            guessInput.value = '';
            guessInput.focus();
        }

        function resetGame() {
            // 重置游戏状态
            secretNumber = Math.floor(Math.random() * 100) + 1;
            attempts = 0;
            
            // 重置界面
            document.getElementById('guessInput').disabled = false;
            document.getElementById('guessInput').value = '';
            document.getElementById('result').innerHTML = '';
            document.getElementById('attempts').innerHTML = '';
            
            // 聚焦到输入框
            document.getElementById('guessInput').focus();
        }

        // 页面加载完成后聚焦到输入框
        window.onload = function() {
            document.getElementById('guessInput').focus();
        };

        // 支持按回车键提交
        document.getElementById('guessInput').addEventListener('keyup', function(event) {
            if (event.key === 'Enter') {
                checkGuess();
            }
        });
    </script>
</body>
</html>
'''

# 将HTML代码保存到文件
with open(r'C:\Users\19716\Desktop\guess_number_game.html', 'w', encoding='utf-8') as f:
    f.write(html_code)

print("简单的猜数字游戏HTML页面已创建并保存到桌面: guess_number_game.html")
print("这个游戏展示了使用AI辅助编程工具可以快速创建一个交互式网页应用。")
print("主要功能包括：")
print("1. 随机生成1-100之间的数字")
print("2. 用户输入猜测并获得反馈")
print("3. 限制最多10次尝试机会")
print("4. 显示尝试次数和结果")
print("5. 支持重新开始游戏")
print("6. 支持键盘回车提交")